# Chroma Vector Store - MMR Retrieval for Policy Document Q&A

This notebook demonstrates how to use **Maximum Marginal Relevance (MMR)** retrieval with ChromaDB and LlamaIndex for question answering over policy documents.

MMR is a retrieval strategy that balances two goals:
- **Relevance**: retrieved chunks should be relevant to the query
- **Diversity**: retrieved chunks should not be near-duplicates of each other

This is particularly useful for **policy documents**, **legal text**, and **technical manuals** — where the same information is often repeated across multiple sections, and pure similarity search may return redundant chunks.

### What this notebook covers
1. Building a persistent Chroma index from policy documents
2. Comparing similarity search vs MMR retrieval side by side
3. Running a full RAG Q&A pipeline using MMR
4. Guidance on when to use MMR vs similarity search

> **Note**: This notebook complements [ChromaIndexDemo.ipynb](ChromaIndexDemo.ipynb), which covers basic Chroma setup, persistence, and CRUD operations.

## Setup

If you're opening this Notebook on colab, you will probably need to install LlamaIndex 🦙.

In [ ]:
%pip install llama-index-vector-stores-chroma
%pip install llama-index-embeddings-huggingface
!pip install llama-index

In [ ]:
# import
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from IPython.display import Markdown, display
import chromadb

# set up OpenAI
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")
import openai

openai.api_key = os.environ["OPENAI_API_KEY"]

In [ ]:
# define embedding function
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-base-en-v1.5")

## 1. Create Policy Document Nodes

We create a set of realistic workplace policy document chunks as `TextNode` objects.

Notice that several chunks cover overlapping topics — for example, expense submission deadlines appear in multiple sections. This is typical of real policy documents and is exactly where MMR adds value: it avoids returning near-duplicate chunks and gives the LLM a more diverse context to answer from.

In [ ]:
# Policy document chunks — realistic HR/workplace policy text.
# Several chunks intentionally cover overlapping topics
# to demonstrate MMR's diversity benefit.

nodes = [
    TextNode(
        text=(
            "Employees must submit all expense claims within 30 days of "
            "incurring the cost. Claims submitted after this deadline will "
            "not be reimbursed without written approval from a director."
        ),
        metadata={"source": "expenses_policy", "section": "submission_deadlines"},
    ),
    TextNode(
        text=(
            "All expense claims require original receipts for purchases "
            "over £25. Digital receipts are accepted. Claims without "
            "receipts will be returned to the employee for resubmission."
        ),
        metadata={"source": "expenses_policy", "section": "receipts"},
    ),
    TextNode(
        text=(
            "Expense claims must be submitted within 30 days. Late "
            "submissions require line manager sign-off in addition to "
            "the standard approval process."
        ),
        metadata={"source": "expenses_policy", "section": "late_submissions"},
    ),
    TextNode(
        text=(
            "All expense claims must be approved by the employee's line "
            "manager before reimbursement is processed. Claims submitted "
            "without manager approval will be automatically rejected."
        ),
        metadata={"source": "expenses_policy", "section": "approval_process"},
    ),
    TextNode(
        text=(
            "Remote working employees are entitled to claim broadband "
            "costs up to £30 per month. Office equipment purchases up to "
            "£500 per year are also reimbursable with prior approval."
        ),
        metadata={"source": "remote_work_policy", "section": "home_office_expenses"},
    ),
    TextNode(
        text=(
            "Travel expenses for client visits are reimbursed at the "
            "standard rail rate. First class travel requires director "
            "approval and must be justified in writing."
        ),
        metadata={"source": "travel_policy", "section": "client_travel"},
    ),
    TextNode(
        text=(
            "Hotel accommodation for overnight business travel is "
            "reimbursed up to £150 per night in the UK and £200 per "
            "night internationally. Bookings must be made via the "
            "company travel portal."
        ),
        metadata={"source": "travel_policy", "section": "accommodation"},
    ),
    TextNode(
        text=(
            "Annual leave must be requested via the HR portal with a "
            "minimum of two weeks notice. Requests are subject to "
            "manager approval based on team capacity."
        ),
        metadata={"source": "leave_policy", "section": "annual_leave"},
    ),
    TextNode(
        text=(
            "Sick leave should be reported to the line manager by 9am "
            "on the first day of absence. A fit note from a GP is "
            "required for absences exceeding seven consecutive days."
        ),
        metadata={"source": "leave_policy", "section": "sick_leave"},
    ),
    TextNode(
        text=(
            "Business meals and entertainment expenses are reimbursable "
            "up to £50 per person. All claims must include the names of "
            "attendees and the business purpose of the meeting."
        ),
        metadata={"source": "expenses_policy", "section": "meals_entertainment"},
    ),
]

## 2. Build Chroma Index with Persistence

We use `PersistentClient` so the index is saved to disk and can be reloaded across sessions.

In [ ]:
# create a persistent Chroma client and collection
db = chromadb.PersistentClient(path="./chroma_policy_db")
chroma_collection = db.get_or_create_collection("company_policies")

# set up ChromaVectorStore and StorageContext
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# build an empty index, then insert nodes
index = VectorStoreIndex(
    [],
    storage_context=storage_context,
    embed_model=embed_model,
)
index.insert_nodes(nodes)

print(f"Index built with {len(nodes)} policy document chunks")

## 3. Similarity Search vs MMR — Side by Side

Here we demonstrate the core difference between the two retrieval modes.

The query `"What are the rules for submitting expense claims?"` will likely retrieve multiple chunks about expense submission — because several policy sections cover overlapping content about deadlines and approval.

- **Similarity search** returns the most relevant chunks, which may be near-duplicates
- **MMR** returns relevant chunks while actively penalising redundancy

In [ ]:
query = "What are the rules for submitting expense claims?"

# --- Similarity search ---
retriever_similarity = index.as_retriever(
    similarity_top_k=3,
)
similarity_results = retriever_similarity.retrieve(query)

print("=" * 60)
print("SIMILARITY SEARCH RESULTS")
print("=" * 60)
for i, node in enumerate(similarity_results):
    print(f"\nResult {i+1} [section: {node.metadata.get('section')}]")
    print(node.text)

In [ ]:
# --- MMR retrieval ---
# vector_store_query_mode="mmr" activates Maximum Marginal Relevance.
# MMR penalises chunks that are too similar to already-selected results,
# promoting diversity in the retrieved context.
#
# Note: mmr_threshold is not currently supported by ChromaVectorStore.
# Use similarity_top_k to control the number of returned results.
# See: https://github.com/run-llama/llama_index/issues/10682

retriever_mmr = index.as_retriever(
    vector_store_query_mode="mmr",
    similarity_top_k=3,
)
mmr_results = retriever_mmr.retrieve(query)

print("=" * 60)
print("MMR RETRIEVAL RESULTS")
print("=" * 60)
for i, node in enumerate(mmr_results):
    print(f"\nResult {i+1} [section: {node.metadata.get('section')}]")
    print(node.text)

**Expected observation**: Similarity search tends to return multiple chunks about submission deadlines, since they score most similarly to the query. MMR returns a more diverse set — covering deadlines, receipts, and approval — giving the LLM broader context to answer from.

## 4. Full RAG Q&A Pipeline with MMR

Now we build a complete query engine that uses MMR retrieval and passes the diverse context to an LLM for answer generation.

In [ ]:
# build a query engine with MMR retrieval
query_engine = index.as_query_engine(
    vector_store_query_mode="mmr",
    similarity_top_k=3,
)

# ask questions about the policy documents
questions = [
    "What is the deadline for submitting expense claims?",
    "What expenses can remote workers claim?",
    "What are the rules for business travel accommodation?",
]

for question in questions:
    print(f"\nQ: {question}")
    response = query_engine.query(question)
    display(Markdown(f"**A:** {response}"))
    print("-" * 60)

## 5. Loading a Persisted Index

If you have already built the index and saved it to disk, you can reload it without re-embedding your documents.

In [ ]:
# load from disk
db2 = chromadb.PersistentClient(path="./chroma_policy_db")
chroma_collection2 = db2.get_or_create_collection("company_policies")
vector_store2 = ChromaVectorStore(chroma_collection=chroma_collection2)

# load index from the existing vector store — no re-embedding needed
index2 = VectorStoreIndex.from_vector_store(
    vector_store2,
    embed_model=embed_model,
)

# query using MMR
query_engine2 = index2.as_query_engine(
    vector_store_query_mode="mmr",
    similarity_top_k=3,
)

response2 = query_engine2.query(
    "Do I need manager approval to claim expenses?"
)
display(Markdown(f"**A:** {response2}"))

## 6. When to Use MMR vs Similarity Search

| Scenario | Recommended mode |
|---|---|
| Knowledge base has overlapping or repeated content (policy docs, legal text, FAQs) | **MMR** |
| You want the most directly relevant chunks regardless of overlap | **Similarity** |
| Documents are diverse and non-overlapping | **Similarity** |
| You want varied context passed to the LLM to reduce hallucination risk | **MMR** |
| Latency is critical and document content is well-structured | **Similarity** |

### Key parameter
- `similarity_top_k` — controls how many documents are returned. Increase for broader context, decrease for more focused answers.

### Note on `mmr_threshold`
The `mmr_threshold` parameter controls the relevance/diversity trade-off in LlamaIndex's built-in in-memory vector store, but is **not currently supported** by `ChromaVectorStore`. See [issue #10682](https://github.com/run-llama/llama_index/issues/10682) for updates.